# Load Data

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

# TODO: Define columns to use as time-series features
# TODO: Normalization
TS_FEATURE_COLS = [
    "tbl_speed",
    "fom",
    "main_comp",
    "tbl_fill",
    "SREL",
    "pre_comp",
    "produced",
    "waste",
    "cyl_main",
    "cyl_pre",
    "stiffness",
    "ejection"
]

TARGET_COLS = ["total_waste", "Total impurities"]

# Load process-level targets
print("Loading Process.csv (targets)...")
process = pd.read_csv("data/Process.csv", sep=";")

# TODO: Keep only the necessary columns
cols = ["batch", "code"] + TARGET_COLS
process = process[cols]

# TODO: Drop rows with NaN in either target
process = process.dropna(subset=TARGET_COLS).reset_index(drop=True)
print(f"Process targets shape: {process.shape}")

# TODO: Load and concatenate all time-series CSV files
print("Loading time-series CSV files...")
all_paths = sorted(
    p for p in glob.glob(os.path.join("data", "*.csv"))
    if os.path.basename(p) != "Process.csv"
)

dfs = []
for path in all_paths:
    df = pd.read_csv(path, sep=";")
    dfs.append(df)

ts = pd.concat(dfs, ignore_index=True)

# Convert timestamp to datetime and sort in correct temporal order
ts["timestamp"] = pd.to_datetime(ts["timestamp"])
ts = ts.sort_values(["code", "timestamp"]).reset_index(drop=True)
print(f"Time-series shape: {ts.shape}")


X_train shape: (789, 34, 1), y_train shape: (789, 2)
X_val shape: (198, 34, 1), y_val shape: (198, 2)


# Build Sequences

In [ ]:
# Restrict to batches that have valid targets
valid_batches = set(process["batch"].unique())
ts = ts[ts["batch"].isin(valid_batches)].copy()

# Fill missing values in feature columns within each batch
def fill_group(g):
    g = g.copy()
    # Forward-fills and backward-fills within each batch to handle NaNs
    g[TS_FEATURE_COLS] = g[TS_FEATURE_COLS].ffill().bfill().fillna(0.0)
    return g

ts = ts.groupby("batch", group_keys=False).apply(fill_group)

# Compute per-batch sequence lengths
batch_lengths = ts.groupby("batch").size()

# Define a max sequence length (95th percentile)
q95 = int(np.quantile(batch_lengths.values, 0.95))
max_seq_len = int(min(5000, max(q95, 1)))
print(f"Using max sequence length: {max_seq_len}")

# Prepare arrays
batches_sorted = sorted(batch_lengths.index.tolist())

X_list, y_list = [], []
for b in batches_sorted:
    g = ts[ts["batch"] == b]
    # Extract feature matrix (T, C)
    seq = g[TS_FEATURE_COLS].values.astype(np.float32)
    
    # Truncate or pad to maximum sequence length
    if seq.shape[0] >= max_seq_len:
        seq = seq[:max_seq_len, :]
    else:
        pad_len = max_seq_len - seq.shape[0]
        pad = np.zeros((pad_len, seq.shape[1]), dtype=np.float32)
        seq = np.vstack([seq, pad])
        
    # Align each batch with its target values
    row = process.loc[process["batch"] == b, TARGET_COLS]
    y = row.iloc[0].values.astype(np.float32)
    
    X_list.append(seq)
    y_list.append(y)

X = np.stack(X_list, axis=0)    # (num_batches, T, C)
y = np.stack(y_list, axis=0)    # (num_batches, 2)

print(f"Built sequences: X shape: {X.shape}, y shape: {y.shape}")


# Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

num_batches, T, C = X.shape

# Scale features per-channel over all timesteps and batches
X_reshaped = X.reshape(-1, C)  # (num_batches * T, C)
x_scaler = StandardScaler()
X_scaled_reshaped = x_scaler.fit_transform(X_reshaped)
X_scaled = X_scaled_reshaped.reshape(num_batches, T, C).astype(np.float32)

# Scale targets over the 2D target array
y_scaler = StandardScaler()
y_scaled = y_scaler.fit_transform(y).astype(np.float32)

# Train/validation split at batch level
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")



Training Epoch 1/100
7/7 - loss: 1.8111 - MAE: 0.9448 - val_loss: 1.0294 - val_accuracy: 0.7944
Learning rate: 0.001000
Validation accuracy improved to 0.794432
Updated the best model

Training Epoch 2/100
7/7 - loss: 1.0723 - MAE: 0.7836 - val_loss: 1.0214 - val_accuracy: 0.7944
Learning rate: 0.001000
No improvement (patience: 1/5)

Training Epoch 3/100
7/7 - loss: 1.0310 - MAE: 0.7728 - val_loss: 1.0133 - val_accuracy: 0.7922
Learning rate: 0.000999
Validation accuracy improved to 0.792241
Updated the best model

Training Epoch 4/100
7/7 - loss: 1.0077 - MAE: 0.7760 - val_loss: 1.0093 - val_accuracy: 0.7904
Learning rate: 0.000998
Validation accuracy improved to 0.790417
Updated the best model

Training Epoch 5/100
7/7 - loss: 0.9170 - MAE: 0.7425 - val_loss: 1.0047 - val_accuracy: 0.7884
Learning rate: 0.000996
Validation accuracy improved to 0.788409
Updated the best model

Training Epoch 6/100
7/7 - loss: 0.8580 - MAE: 0.7271 - val_loss: 1.0035 - val_accuracy: 0.7856
Learning ra

7/7 - loss: 0.3865 - MAE: 0.4536 - val_loss: 0.5029 - val_accuracy: 0.4972
Learning rate: 0.000563
No improvement (patience: 3/5)

Training Epoch 48/100
7/7 - loss: 0.5021 - MAE: 0.4615 - val_loss: 0.4854 - val_accuracy: 0.4882
Learning rate: 0.000547
Validation accuracy improved to 0.488214
Updated the best model

Training Epoch 49/100
7/7 - loss: 0.4759 - MAE: 0.4474 - val_loss: 0.4691 - val_accuracy: 0.4730
Learning rate: 0.000531
Validation accuracy improved to 0.472986
Updated the best model

Training Epoch 50/100
7/7 - loss: 0.3841 - MAE: 0.4500 - val_loss: 0.4721 - val_accuracy: 0.4697
Learning rate: 0.000516
Validation accuracy improved to 0.469671
Updated the best model

Training Epoch 51/100
7/7 - loss: 0.3839 - MAE: 0.4521 - val_loss: 0.4721 - val_accuracy: 0.4679
Learning rate: 0.000500
Validation accuracy improved to 0.467896
Updated the best model

Training Epoch 52/100
7/7 - loss: 0.5095 - MAE: 0.4576 - val_loss: 0.4863 - val_accuracy: 0.4746
Learning rate: 0.000484
No i

# Build and Train CNN

In [ ]:
from utils.cnn import CNN_trainer

import importlib
import utils.cnn as cnn
importlib.reload(cnn)

# Create a random number generator with a fixed seed
rng = np.random.default_rng(42)

# Training and validation spilt (0.2)
train_idx, val_idx = [], []

# Iterate through each class label (0, 1, 2, 3, 4)
for c in range(5):
    # Identify all samples that belong to class c
    class_idx = np.where(y_train == c)[0]
    # Randomly shuffle the indices
    class_idx = rng.permutation(class_idx)
    # Split the dataset
    split = int(len(class_idx) * 0.2)
    train_idx.extend(class_idx[split:])
    val_idx.extend(class_idx[:split])

train_idx = np.array(train_idx)
val_idx = np.array(val_idx)

X_t, X_v = X_train, X_val
y_t, y_v = y_train, y_val

# Build and train the model
trainer = CNN_trainer(X_t, y_t, X_v, y_v,
    epochs=100,      # Number of epochs
    batch_size=128, # Batch size
    lr=1e-3,        # Learning rate
    l2=1e-5,        # L2 regularization coefficient
    dropout=0.3     # Dropout rate for regularization
)
trainer.run()


# Build and Train LSTM

In [ ]:
import importlib
import utils.lstm as lstm
importlib.reload(lstm)

from utils.lstm import LSTM_trainer

#X_t, y_t, X_v, y_v are the same shapes as for CNN_trainer

lstm_trainer = LSTM_trainer(X_t, y_t, X_v, y_v, epochs=100, batch_size=128, lr=1e-3,
l2=1e-5, dropout=0.3)
lstm_trainer.run()


# Save the Best Model